In [1]:
!pip install -U bitsandbytes>=0.46.1

#Saving & Loading the model

In [2]:

import os
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig)
import torch
import bitsandbytes as bnb

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model_path = (
    "rag"
    "qwen2.5-1.5b-instruct"
)

if not os.path.exists(model_path):
  model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, quantization_config=bnb_config, device_map="auto")
  model.save_pretrained(model_path)
  tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
  tokenizer.save_pretrained(model_path)
else:
  model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True, quantization_config=bnb_config, device_map="auto")
  tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

#Tools

In [13]:
# Calculator

def calculator(expression: str) -> dict:
    try:
        result = eval(expression,{"__builtins__": {}},{})
        return {
            "result": result,
            "status": "success"
        }
    except Exception as e:
        return {
            "result": None,
            "status": f"error + {str(e)}"
        }

def unit_converter(value, from_unit, to_unit):

    conversions = {
        ("km", "miles"): 0.621371,
        ("miles", "km"): 1.60934,
        ("kg", "g"): 1000,
        ("g", "kg"): 0.001,
        ("m", "cm"): 100,
        ("cm", "m"): 0.01
    }

    key = (from_unit.lower(), to_unit.lower())

    if key not in conversions:
        return {
            "status": "error",
            "result": None,
            "error": "Unsupported unit conversion."
        }

    result = value * conversions[key]

    return {
        "status": "success",
        "result": result,
        "from_unit": from_unit,
        "to_unit": to_unit
    }

from google.colab import userdata
import requests

def web_search(query):

    api_key = userdata.get("TAVILY_API_KEY")

    if not api_key:
        return {
            "status": "error",
            "error": "TAVILY_API_KEY is not configured."
        }

    try:

        response = requests.post(
            "https://api.tavily.com/search",

            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },

            json={
                "query": query
            },

            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        results = []

        for item in data.get("results", [])[:5]:

            content = item.get("content", "")

            content = content[:200]

            results.append({
                "title": item.get("title"),
                "url": item.get("url"),
                "content": content,
                "score": item.get("score")
            })

        return {
            "status": "success",
            "query": query,
            "results": results
        }

    except requests.exceptions.Timeout:

        return {
            "status": "error",
            "error": "Search request timed out."
        }

    except requests.exceptions.RequestException as e:

        return {
            "status": "error",
            "error": str(e)
        }

In [12]:
## testing web_search
# ans = web_search("What is agentic ai")
# if ans["status"] == "success":
#   print(ans["results"])
#   print(len(ans["results"]))
# else:
#   print(ans["error"])

[{'title': 'What is Agentic AI? | UiPath', 'url': 'https://www.uipath.com/ai/agentic-ai', 'content': 'Agentic AI is a new form of AI that enables agents to act autonomously to pursue goals, performing complex, decision-intensive workflows that until recently were considered too dynamic, contextual, and consequential to be automated. [...] The possibilities are virtually endless, and the future of agentic AI is filled with promise. As this technology evolves, it is reshaping the world of work and the roles of humans and machines in the world.\n\nWhat is agentic AI in simple terms?\n\nAgentic AI is the intelligence that enables AI agents to understand context, make decisions, and take action to achieve goals. It allows AI to operate beyond single prompts and perform multistep work autonomously. [...] Agentic AI, in short, is enabling us to explore entirely new possibilities in designing work processes, expanding the role of intelligent orchestration and automation as we redefine the role

#2 LLM Calls

In [14]:
tools = [
    {
        "name": "calculator",
        "description": (
            "Use this tool when the user requires an exact mathematical "
            "calculation. Do not use it for conceptual questions about math."
        ),
        "schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression to calculate."
                }
            },
            "required": ["expression"]
        }
    },
    # {
    # "name": "unit_converter",
    # "description": (
    #     "Use this tool when the user asks to convert a quantity "
    #     "from one unit to another, such as length, weight, temperature, "
    #     "time, area, volume, or speed. Do not use it for general "
    #     "mathematical calculations."
    # ),
    # "schema": {
    #     "type": "object",
    #     "properties": {
    #         "value": {
    #             "type": "number",
    #             "description": "The numerical value to convert."
    #         },
    #         "from_unit": {
    #             "type": "string",
    #             "description": "The unit being converted from."
    #         },
    #         "to_unit": {
    #             "type": "string",
    #             "description": "The unit being converted to."
    #         }
    #     },
    #     "required": ["value", "from_unit", "to_unit"]
    # }
    # }, # the problem ut came across is llm is making a call with gram to kilogram, but here only kg, g is presnt which makes the function call to invalid
         #  {"tools_required": true,"tool_name": "unit_converter","arguments": {"value": 100, "from_unit": "gram", "to_unit": "kilogram"}}, VALIDATION: Unknown tool. Tool call rejected.
    {
      "name": "unit_converter",
      "description": (
          "Convert a numerical value between supported units. "
          "Use the exact unit codes specified in the parameter descriptions."
      ),
      "schema": {
          "type": "object",
          "properties": {
              "value": {
                  "type": "number",
                  "description": "The numerical value to convert."
              },

              "from_unit": {
                  "type": "string",
                  "description": (
                      "Source unit. Use ONLY one of these exact unit codes: "
                      "g, kg, m, cm, km, miles."
                  )
              },

              "to_unit": {
                  "type": "string",
                  "description": (
                      "Target unit. Use ONLY one of these exact unit codes: "
                      "g, kg, m, cm, km, miles."
                  )
              }
          },

          "required": [
              "value",
              "from_unit",
              "to_unit"
          ]
      }
    },
    {
      "name": "web_search",
      "description": (
          "Use this tool when the user asks for current, recent, "
          "up-to-date, or externally available information that may "
          "not be reliably known by the language model. "
          "Do not use it for questions that can be answered from "
          "general knowledge without external information."
      ),
      "schema": {
          "type": "object",
          "properties": {
              "query": {
                  "type": "string",
                  "description": "The search query to send to the web search engine."
              }
          },
          "required": ["query"]
      }
    }
]

def call_llm(messages, max_new_tokens=128):

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    # Only get newly generated tokens.
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

def decide_tool(question, tools):

    prompt = [
        {
            "role": "system",
            "content": """
You are a tool-selection system.
Decide whether the user's question requires one of the available tools.
If no tool is required, return ONLY valid JSON:
{"tools_required": false}
If a tool is required, return ONLY valid JSON:
{"tools_required": true,"tool_name": "calculator","arguments": {"expression": "..."}}
Do not answer the user's question.
Do not include markdown.
Do not include explanations.
"""
        },
        {
            "role": "user",
            "content": f"""
User question:
{question}
Available tools:
{tools}
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    # Only decode the newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response


#Validation

In [15]:
import json

def validate_tool_call(decision):

    # -------------------------
    # Basic structure
    # -------------------------

    if not isinstance(decision, dict):
        return False, "Decision must be a dictionary."

    if "tools_required" not in decision:
        return False, "Missing tools_required."

    if not isinstance(decision["tools_required"], bool):
        return False, "tools_required must be a boolean."

    # -------------------------
    # No tool
    # -------------------------

    if decision["tools_required"] is False:
        return True, "Valid: no tool required."

    # -------------------------
    # Tool name
    # -------------------------

    if "tool_name" not in decision:
        return False, "Missing tool_name."

    allowed_tools = [
        "calculator",
        "unit_converter",
        "web_search"
    ]

    tool_name = decision["tool_name"]

    if tool_name not in allowed_tools:
        return False, f"Unknown tool: {tool_name}"

    # -------------------------
    # Arguments
    # -------------------------

    if "arguments" not in decision:
        return False, "Missing arguments."

    arguments = decision["arguments"]

    if not isinstance(arguments, dict):
        return False, "Arguments must be a dictionary."

    # ==================================================
    # CALCULATOR
    # ==================================================

    if tool_name == "calculator":

        if "expression" not in arguments:
            return False, "Missing expression."

        if not isinstance(arguments["expression"], str):
            return False, "expression must be a string."

        return True, "Valid calculator tool call."

    # ==================================================
    # UNIT CONVERTER
    # ==================================================

    if tool_name == "unit_converter":

        required = [
            "value",
            "from_unit",
            "to_unit"
        ]

        for field in required:
            if field not in arguments:
                return False, f"Missing {field}."

        if not isinstance(arguments["value"], (int, float)):
            return False, "value must be a number."

        if not isinstance(arguments["from_unit"], str):
            return False, "from_unit must be a string."

        if not isinstance(arguments["to_unit"], str):
            return False, "to_unit must be a string."

        allowed_units = [
            "g",
            "kg",
            "m",
            "cm",
            "km",
            "miles"
        ]

        if arguments["from_unit"] not in allowed_units:
            return False, (
                f"Unsupported from_unit: "
                f"{arguments['from_unit']}"
            )

        if arguments["to_unit"] not in allowed_units:
            return False, (
                f"Unsupported to_unit: "
                f"{arguments['to_unit']}"
            )

        return True, "Valid unit_converter tool call."

    # ==================================================
    # WEB SEARCH
    # ==================================================

    if tool_name == "web_search":

        if "query" not in arguments:
            return False, "Missing query."

        if not isinstance(arguments["query"], str):
            return False, "query must be a string."

        if not arguments["query"].strip():
            return False, "query cannot be empty."

        return True, "Valid web_search tool call."

    return False, "Validation failed."

In [17]:
def generate_final_answer(
    question,
    tool_decision=None,
    tool_result=None
):

    # --------------------------------------------------------
    # CASE 1:
    # No tool was required
    # --------------------------------------------------------

    if tool_decision is not None and \
       tool_decision["tools_required"] is False:

        messages = [

            {
                "role": "system",

                "content": """
Answer the user's question directly.

No external tool is required.
Give a clear and concise answer.
"""
            },

            {
                "role": "user",

                "content": question
            }

        ]

    # --------------------------------------------------------
    # CASE 2:
    # Tool was used
    # --------------------------------------------------------

    else:

        messages = [

            {
                "role": "system",

                "content": """
Answer the user's original question using
the tool result provided below.

Do not perform the calculation yourself.

Use the tool result as the authoritative result.

Give only the final natural-language answer.
"""
            },

            {
                "role": "user",

                "content": question
            },

            {
                "role": "assistant",

                "content": json.dumps(
                    tool_decision
                )
            },

            {
                "role": "tool",

                "content": json.dumps(
                    tool_result
                )
            }

        ]

    return call_llm(
        messages,
        max_new_tokens=128
    )


# ============================================================
# 8. COMPLETE TOOL-CALLING PIPELINE
# ============================================================

def run_agent(question):

    print("\nUSER:")
    print(question)

    # --------------------------------------------------------
    # STEP 1
    # LLM decides whether tool is required
    # --------------------------------------------------------

    raw_decision = decide_tool(question, tools)

    print("\nLLM #1 RAW OUTPUT:")
    print(raw_decision)


    # --------------------------------------------------------
    # STEP 2
    # Parse JSON
    # --------------------------------------------------------

    try:

        decision = json.loads(raw_decision)

    except json.JSONDecodeError:

        print("\nERROR: LLM returned invalid JSON.")

        return


    print("\nPARSED DECISION:")
    print(json.dumps(
        decision,
        indent=2
    ))


    # --------------------------------------------------------
    # STEP 3
    # Validate
    # --------------------------------------------------------

    valid, message = validate_tool_call(
        decision
    )

    print("\nVALIDATION:")
    print(message)


    if not valid:

        print("Tool call rejected.")

        return


    # --------------------------------------------------------
    # STEP 4
    # NO TOOL
    # --------------------------------------------------------

    if decision["tools_required"] is False:

        print("\nNO TOOL REQUIRED")

        final_answer = generate_final_answer(
            question,
            tool_decision=decision
        )

        print("\nFINAL ANSWER:")
        print(final_answer)

        return final_answer


    # --------------------------------------------------------
    # STEP 5
    # TOOL REQUIRED
    # --------------------------------------------------------

    print("\nTOOL REQUIRED")

    tool_name = decision["tool_name"]

    arguments = decision["arguments"]

    print("\nSELECTED TOOL:")
    print(tool_name)

    print("\nTOOL ARGUMENTS:")
    print(arguments)


    # --------------------------------------------------------
    # STEP 6
    # Execute tool
    # --------------------------------------------------------

    if tool_name == "calculator":

      tool_result = calculator(
          arguments["expression"]
      )

    elif tool_name == "unit_converter":

      tool_result = unit_converter(
          arguments["value"],
          arguments["from_unit"],
          arguments["to_unit"]
      )

    elif tool_name == "web_search":

      tool_result = web_search(
          arguments["query"]
      )

    else:

        print("Unknown tool.")
        return


    print("\nTOOL RESULT:")
    print(tool_result)


    # --------------------------------------------------------
    # STEP 7
    # Send tool result back to LLM
    # --------------------------------------------------------

    final_answer = generate_final_answer(
        question,
        tool_decision=decision,
        tool_result=tool_result
    )

    print("\nFINAL ANSWER:")
    print(final_answer)

    return final_answer


In [19]:
question = input("\nAsk: ")

run_agent(question)


Ask: what is current population of india and china, and how many percentage india is ahead of china in population?

USER:
what is current population of india and china, and how many percentage india is ahead of china in population?

LLM #1 RAW OUTPUT:
{"tools_required": true,"tool_name": "calculator","arguments": {"expression": "India_population = 1387000000; China_population = 1426500000; India_percentage_ahead = (India_population - China_population) / China_population * 100; India_percentage_ahead"}}

PARSED DECISION:
{
  "tools_required": true,
  "tool_name": "calculator",
  "arguments": {
    "expression": "India_population = 1387000000; China_population = 1426500000; India_percentage_ahead = (India_population - China_population) / China_population * 100; India_percentage_ahead"
  }
}

VALIDATION:
Valid calculator tool call.

TOOL REQUIRED

SELECTED TOOL:
calculator

TOOL ARGUMENTS:
{'expression': 'India_population = 1387000000; China_population = 1426500000; India_percentage_ahea

'The current population of India is approximately 1.39 billion people, while the population of China is about 1.43 billion people. India is currently ahead of China by around 36 million people.'